In [1]:
# =============================================================================
# Ablation: Re-uploading Depth for Indian Pines
#
# Tests N_REUP_LAYERS ∈ {2, 3, 4, 5} with winning LR (0.0005) and N_QUBITS=4.
# 2 seeds per config on VAL. Reports OA/AA/κ.
#
# Output: Ablation table + winner → use in Phase 2c (beta ablation)
# =============================================================================

import time
GLOBAL_START = time.time()

def elapsed():
    return time.time() - GLOBAL_START

print("[Setup] Importing libraries...")
t0 = time.time()

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml
from scipy.io import loadmat
from sklearn.metrics import accuracy_score, cohen_kappa_score
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"[Setup] GPU detected: {torch.cuda.get_device_name(0)}")

print(f"[Setup] Device: {DEVICE}")
print(f"[Setup] Completed in {time.time()-t0:.2f}s | Total: {elapsed():.2f}s")

# =============================================================================
# ABLATION CONFIGURATION
# =============================================================================
DEPTH_CONFIGS = [
    {"name": "2l", "N_REUP_LAYERS": 2},
    {"name": "3l", "N_REUP_LAYERS": 3},
    {"name": "4l", "N_REUP_LAYERS": 4},
    {"name": "5l", "N_REUP_LAYERS": 5},
]
ABLATION_SEEDS = [42, 0]

# Fixed from previous phases
N_QUBITS = 4
LR_QUANTUM_WINNER = 0.0005

print(f"\n[Ablation] Re-uploading layers: {[c['N_REUP_LAYERS'] for c in DEPTH_CONFIGS]}")
print(f"[Ablation] N_QUBITS: {N_QUBITS}")
print(f"[Ablation] LR_QUANTUM: {LR_QUANTUM_WINNER}")
print(f"[Ablation] Seeds: {ABLATION_SEEDS}")
print(f"[Ablation] Total runs: {len(DEPTH_CONFIGS) * len(ABLATION_SEEDS)}")

# =============================================================================
# DATASET LOADING
# =============================================================================
print("\n" + "=" * 60)
print("[Data] Loading Indian Pines...")
t0 = time.time()

DATA_PATH = r"indian pines data\Indian_pines_corrected.mat"
GT_PATH = r"indian pines data\Indian_pines_gt.mat"

data_mat = loadmat(DATA_PATH)
gt_mat = loadmat(GT_PATH)
data_keys = [k for k in data_mat.keys() if not k.startswith("__")]
gt_keys = [k for k in gt_mat.keys() if not k.startswith("__")]
data_raw = data_mat[data_keys[0]].astype(np.float32)
gt = gt_mat[gt_keys[0]].astype(np.int64)

H, W, N_BANDS = data_raw.shape
unique_labels = np.sort(np.unique(gt))
unique_labels = unique_labels[unique_labels > 0]
NUM_CLASSES = len(unique_labels)
LABEL_MAP = {orig: idx for idx, orig in enumerate(unique_labels)}

print(f"[Data] Image: ({H}, {W}, {N_BANDS}), Classes: {NUM_CLASSES}")

# =============================================================================
# FIXED HYPERPARAMETERS
# =============================================================================
N_SUB_ROTATIONS = 3
ROUTER_BOTTLENECK = 32
SPATIAL_DIM = 32
FUSION_DIM = 64
DROPOUT = 0.2
PATCH_SIZE = 9
TRAIN_SAMPLES_PER_CLASS = 30
VAL_SAMPLES_PER_CLASS = 15
HALF = PATCH_SIZE // 2

BATCH_SIZE = 16
N_EPOCHS = 50
LR_CLASSICAL = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0
TOPOLOGY_BETA = 0.05
FOCAL_GAMMA = 2.0

# =============================================================================
# PREPROCESSING
# =============================================================================
print("\n" + "=" * 60)
print("[Preprocess] Normalizing and extracting patches...")
t0 = time.time()

data_norm = np.zeros_like(data_raw)
for b in range(N_BANDS):
    band = data_raw[:, :, b]
    bmin, bmax = band.min(), band.max()
    if bmax - bmin > 0:
        data_norm[:, :, b] = (band - bmin) / (bmax - bmin)

data_padded = np.pad(data_norm, ((HALF, HALF), (HALF, HALF), (0, 0)), mode="reflect")
labeled_indices = np.argwhere(gt > 0)
N_labeled = len(labeled_indices)

patches = np.zeros((N_labeled, PATCH_SIZE, PATCH_SIZE, N_BANDS), dtype=np.float32)
spectra = np.zeros((N_labeled, N_BANDS), dtype=np.float32)
labels = np.zeros(N_labeled, dtype=np.int64)

for idx, (i, j) in enumerate(labeled_indices):
    patches[idx] = data_padded[i:i + PATCH_SIZE, j:j + PATCH_SIZE, :]
    spectra[idx] = data_norm[i, j, :]
    labels[idx] = LABEL_MAP[gt[i, j]]

print(f"[Preprocess] Extracted {N_labeled} patches in {time.time()-t0:.2f}s")

# =============================================================================
# QUANTUM CIRCUIT (template)
# =============================================================================
def make_quantum_circuit(n_qubits, n_reup_layers, n_sub_rotations):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(enc_angles, thetas, ent_angles):
        for l in range(n_reup_layers):
            for q in range(n_qubits):
                for s in range(n_sub_rotations):
                    idx = (l * n_qubits * n_sub_rotations + q * n_sub_rotations + s) * 3
                    qml.Rot(
                        thetas[idx]     + enc_angles[idx],
                        thetas[idx + 1] + enc_angles[idx + 1],
                        thetas[idx + 2] + enc_angles[idx + 2],
                        wires=q
                    )
            if l < n_reup_layers - 1:
                for q in range(n_qubits):
                    ent_idx = l * n_qubits + q
                    qml.CRZ(ent_angles[ent_idx], wires=[q, (q + 1) % n_qubits])
        return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]
    return circuit

# =============================================================================
# MODEL CLASSES
# =============================================================================
class AdaptiveQuantumBranch(nn.Module):
    def __init__(self, n_bands, n_qubits, n_reup_layers, n_sub_rotations,
                 bottleneck=32, circuit=None):
        super().__init__()
        self.n_bands = n_bands
        self.n_qubits = n_qubits
        self.n_reup_layers = n_reup_layers
        self.n_sub_rotations = n_sub_rotations
        self.circuit = circuit
        encoding_dim = n_qubits * n_sub_rotations * 3

        self.band_routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_bands, bottleneck),
                nn.ReLU(),
                nn.Linear(bottleneck, n_bands),
                nn.Sigmoid()
            ) for _ in range(n_reup_layers)
        ])
        self.projections = nn.ModuleList([
            nn.Linear(n_bands, encoding_dim) for _ in range(n_reup_layers)
        ])
        self.ent_routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_bands, bottleneck),
                nn.ReLU(),
                nn.Linear(bottleneck, n_qubits),
                nn.Tanh()
            ) for _ in range(n_reup_layers - 1)
        ])
        total_thetas = n_reup_layers * encoding_dim
        self.thetas = nn.Parameter(torch.randn(total_thetas) * 0.1)

    def forward(self, x):
        batch_size = x.shape[0]
        routing_weights, ent_weights = [], []
        all_enc, all_ent = [], []

        for l in range(self.n_reup_layers):
            alpha = self.band_routers[l](x)
            routing_weights.append(alpha)
            z = alpha * x
            enc = self.projections[l](z) * math.pi
            all_enc.append(enc)
            if l < self.n_reup_layers - 1:
                phi = self.ent_routers[l](x) * math.pi
                ent_weights.append(phi)
                all_ent.append(phi)

        flat_enc = torch.cat(all_enc, dim=1)
        flat_ent = torch.cat(all_ent, dim=1)
        q_features = torch.zeros(batch_size, self.n_qubits, device=x.device)
        for i in range(batch_size):
            result = self.circuit(flat_enc[i], self.thetas, flat_ent[i])
            q_features[i] = torch.stack(result)
        return q_features, routing_weights, ent_weights


class SpatialBranch(nn.Module):
    def __init__(self, n_bands, spatial_dim=32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3), stride=(3, 1, 1), padding=(3, 1, 1)),
            nn.BatchNorm3d(8), nn.ReLU(),
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3), stride=(2, 1, 1), padding=(2, 1, 1)),
            nn.BatchNorm3d(16), nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 1, 1), padding=(1, 1, 1)),
            nn.BatchNorm3d(32), nn.ReLU(),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.fc = nn.Linear(32, spatial_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.relu(self.fc(x))


class AdaptiveReuploadClassifier(nn.Module):
    def __init__(self, n_bands, n_qubits, n_reup_layers, n_sub_rotations,
                 n_classes, circuit, router_bottleneck=32, spatial_dim=32,
                 fusion_dim=64, dropout=0.2):
        super().__init__()
        self.quantum_branch = AdaptiveQuantumBranch(
            n_bands, n_qubits, n_reup_layers, n_sub_rotations,
            router_bottleneck, circuit
        )
        self.spatial_branch = SpatialBranch(n_bands, spatial_dim)
        self.q_bn = nn.BatchNorm1d(n_qubits)
        self.s_bn = nn.BatchNorm1d(spatial_dim)
        fusion_input = n_qubits + spatial_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_input, fusion_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, n_classes)
        )

    def forward(self, patches, spectra):
        q_features, routing_weights, ent_weights = self.quantum_branch(spectra)
        s_features = self.spatial_branch(patches)
        q_normed = self.q_bn(q_features)
        s_normed = self.s_bn(s_features)
        fused = torch.cat([q_normed, s_normed], dim=1)
        logits = self.classifier(fused)
        return logits, routing_weights, ent_weights

# =============================================================================
# LOSSES
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        p_t = torch.exp(-ce)
        return ((1.0 - p_t) ** self.gamma * ce).mean()


def topology_diversity_loss(ent_weights, labels, n_classes, beta=0.05):
    if not ent_weights:
        return torch.tensor(0.0, device=labels.device)
    phi = torch.cat(ent_weights, dim=1)
    class_means = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0:
            class_means.append(phi[mask].mean(dim=0))
    if len(class_means) < 2:
        return torch.zeros(1, device=phi.device).squeeze()
    class_means_t = torch.stack(class_means)
    inter_class_var = class_means_t.var(dim=0).mean()
    total_var = phi.var(dim=0).mean().clamp(min=1e-8)
    eta_sq = (inter_class_var / total_var).clamp(max=1.0)
    return -beta * eta_sq


def set_eval_with_bn_train(module):
    module.eval()
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm3d)):
            m.training = True

# =============================================================================
# TRAIN-ONE-RUN FUNCTION
# =============================================================================
def train_one_run(seed, n_reup_layers, circuit):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # Data split
    rng = np.random.RandomState(seed)
    train_sel, val_sel = [], []
    for c in range(NUM_CLASSES):
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)
        n_train = min(TRAIN_SAMPLES_PER_CLASS, max(1, len(class_idx) // 3))
        n_val = min(VAL_SAMPLES_PER_CLASS, max(1, (len(class_idx) - n_train) // 2))
        train_sel.extend(class_idx[:n_train].tolist())
        val_sel.extend(class_idx[n_train:n_train + n_val].tolist())
    train_sel = np.array(train_sel, dtype=np.int64)
    val_sel = np.array(val_sel, dtype=np.int64)
    rng.shuffle(train_sel)
    rng.shuffle(val_sel)

    train_patches_t = torch.FloatTensor(patches[train_sel].transpose(0, 3, 1, 2)[:, np.newaxis])
    val_patches_t   = torch.FloatTensor(patches[val_sel].transpose(0, 3, 1, 2)[:, np.newaxis])
    train_spectra_t = torch.FloatTensor(spectra[train_sel])
    val_spectra_t   = torch.FloatTensor(spectra[val_sel])
    train_labels_t  = torch.LongTensor(labels[train_sel])
    val_labels_t    = torch.LongTensor(labels[val_sel])

    train_ds = TensorDataset(train_patches_t, train_spectra_t, train_labels_t)
    val_ds   = TensorDataset(val_patches_t,   val_spectra_t,   val_labels_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    # Model
    model = AdaptiveReuploadClassifier(
        n_bands=N_BANDS, n_qubits=N_QUBITS,
        n_reup_layers=n_reup_layers, n_sub_rotations=N_SUB_ROTATIONS,
        n_classes=NUM_CLASSES, circuit=circuit,
        router_bottleneck=ROUTER_BOTTLENECK, spatial_dim=SPATIAL_DIM,
        fusion_dim=FUSION_DIM, dropout=DROPOUT
    ).to(DEVICE)

    quantum_params = list(model.quantum_branch.parameters())
    classical_params = (
        list(model.spatial_branch.parameters()) +
        list(model.q_bn.parameters()) +
        list(model.s_bn.parameters()) +
        list(model.classifier.parameters())
    )
    optimizer = optim.Adam([
        {"params": quantum_params,   "lr": LR_QUANTUM_WINNER, "weight_decay": 0},
        {"params": classical_params, "lr": LR_CLASSICAL,      "weight_decay": WEIGHT_DECAY}
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    criterion = FocalLoss(gamma=FOCAL_GAMMA)

    best_val_acc = -1.0
    val_preds_best = None
    val_labels_best = None

    for epoch in range(1, N_EPOCHS + 1):
        ep_start = time.time()
        model.train()
        train_correct, train_total = 0, 0
        for patches_b, spectra_b, labels_b in train_loader:
            patches_b = patches_b.to(DEVICE)
            spectra_b = spectra_b.to(DEVICE)
            labels_b = labels_b.to(DEVICE)

            optimizer.zero_grad()
            logits, _, ent_w = model(patches_b, spectra_b)
            cls_loss = criterion(logits, labels_b)
            topo_loss = topology_diversity_loss(ent_w, labels_b, NUM_CLASSES, TOPOLOGY_BETA)
            loss = cls_loss + topo_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()

            train_correct += (logits.argmax(1) == labels_b).sum().item()
            train_total += labels_b.size(0)
        train_acc = train_correct / train_total

        # Validate
        set_eval_with_bn_train(model)
        val_correct, val_total = 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for patches_b, spectra_b, labels_b in val_loader:
                logits, _, _ = model(patches_b.to(DEVICE), spectra_b.to(DEVICE))
                preds = logits.argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels_b.numpy())
                val_correct += (logits.argmax(1).cpu() == labels_b).sum().item()
                val_total += labels_b.size(0)

        val_acc = val_correct / val_total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            val_preds_best = np.array(all_preds)
            val_labels_best = np.array(all_labels)

        scheduler.step()
        ep_time = time.time() - ep_start
        print(f"    [n_reup={n_reup_layers} seed={seed}] Ep {epoch:02d}/{N_EPOCHS} | "
              f"train={train_acc*100:.1f}% val={val_acc*100:.1f}% | {ep_time:.1f}s",
              end="\r")
    print()

    # Compute metrics on best val
    val_aa = 0.0
    if val_labels_best is not None:
        for c in range(NUM_CLASSES):
            mask = val_labels_best == c
            if mask.sum() > 0:
                val_aa += accuracy_score(val_labels_best[mask], val_preds_best[mask])
        val_aa /= NUM_CLASSES

    val_kappa = cohen_kappa_score(val_labels_best, val_preds_best) if val_labels_best is not None else 0.0

    return best_val_acc, val_aa, val_kappa

# =============================================================================
# ABLATION LOOP
# =============================================================================
print("\n" + "=" * 60)
print("[Ablation] Starting depth ablation...")
ablation_start = time.time()

results = {cfg["name"]: {} for cfg in DEPTH_CONFIGS}

for cfg in DEPTH_CONFIGS:
    n_reup_layers = cfg["N_REUP_LAYERS"]
    circuit = make_quantum_circuit(N_QUBITS, n_reup_layers, N_SUB_ROTATIONS)

    for seed in ABLATION_SEEDS:
        print(f"\n[Ablation] Config={cfg['name']} | Seed={seed}")
        run_start = time.time()
        oa, aa, kappa = train_one_run(seed, n_reup_layers, circuit)
        results[cfg["name"]][seed] = {"OA": oa, "AA": aa, "Kappa": kappa}
        print(f"[Ablation] {cfg['name']} seed={seed} → OA={oa*100:.2f}% AA={aa*100:.2f}% κ={kappa:.4f} | "
              f"elapsed {(time.time()-run_start)/60:.1f}m")

ablation_time = time.time() - ablation_start

# =============================================================================
# RESULTS
# =============================================================================
print("\n" + "=" * 60)
print("[Ablation] RESULTS")
print("=" * 60)
print(f"\n{'Config':<10s} {'Metric':<8s} " +
      " ".join(f"seed={s:>3d}" for s in ABLATION_SEEDS) +
      f"  {'Mean':>7s}  {'Std':>6s}")
print("-" * 70)

config_means = []
for cfg in DEPTH_CONFIGS:
    for metric in ["OA", "AA", "Kappa"]:
        values = [results[cfg["name"]][s][metric] for s in ABLATION_SEEDS]
        mean_val = np.mean(values)
        std_val = np.std(values)
        val_str = " ".join(f"{v*100:>7.2f}%" if metric != "Kappa" else f"{v:>7.4f}" for v in values)
        fmt_mean = f"{mean_val*100:>6.2f}%" if metric != "Kappa" else f"{mean_val:>6.4f}"
        fmt_std = f"{std_val*100:>5.2f}%" if metric != "Kappa" else f"{std_val:>5.4f}"
        print(f"{cfg['name']:<10s} {metric:<8s} {val_str}  {fmt_mean}  {fmt_std}")

    oa_mean = np.mean([results[cfg["name"]][s]["OA"] for s in ABLATION_SEEDS])
    config_means.append((cfg, oa_mean))
    print()

winner = max(config_means, key=lambda x: x[1])
winner_cfg, winner_oa = winner

print("=" * 70)
print(f"[Ablation] WINNER: {winner_cfg['name']}")
print(f"  N_REUP_LAYERS = {winner_cfg['N_REUP_LAYERS']}")
print(f"  Mean val OA = {winner_oa*100:.2f}%")
print("=" * 70)

print(f"\n[Ablation] Paste into Phase 2c (beta ablation):")
print(f"  N_REUP_LAYERS = {winner_cfg['N_REUP_LAYERS']}")

print(f"\n[Ablation] Total time: {ablation_time/60:.1f} min | Wall time: {elapsed()/60:.1f} min")


[Setup] Importing libraries...
[Setup] Device: cpu
[Setup] Completed in 11.25s | Total: 11.25s

[Ablation] Re-uploading layers: [2, 3, 4, 5]
[Ablation] N_QUBITS: 4
[Ablation] LR_QUANTUM: 0.0005
[Ablation] Seeds: [42, 0]
[Ablation] Total runs: 8

[Data] Loading Indian Pines...
[Data] Image: (145, 145, 200), Classes: 16

[Preprocess] Normalizing and extracting patches...
[Preprocess] Extracted 10249 patches in 0.99s

[Ablation] Starting depth ablation...

[Ablation] Config=2l | Seed=42
    [n_reup=2 seed=42] Ep 50/50 | train=96.2% val=86.7% | 18.6s
[Ablation] 2l seed=42 → OA=88.50% AA=89.17% κ=0.8771 | elapsed 22.9m

[Ablation] Config=2l | Seed=0
    [n_reup=2 seed=0] Ep 50/50 | train=96.2% val=88.5% | 18.7s
[Ablation] 2l seed=0 → OA=90.27% AA=90.36% κ=0.8960 | elapsed 15.6m

[Ablation] Config=3l | Seed=42
    [n_reup=3 seed=42] Ep 50/50 | train=92.5% val=86.3% | 23.6s
[Ablation] 3l seed=42 → OA=87.17% AA=87.92% κ=0.8629 | elapsed 20.0m

[Ablation] Config=3l | Seed=0
    [n_reup=3 seed=0